---
title: "Scraping home prices in Ensenada Mexico"
author: "Fabian"
date: "2024-04-01"
categories: ["Data Science"]
image: shot1.png

---

# Discovering the Value of My Home: A Personal Journey into Ensenada's Real Estate Market

Nestled in the heart of Ensenada, Baja Califronia, Mexico is an amazing port city that has always been a part of my life,(since Covid) I've built a home that is not just a place to live but a symbol of my dreams and aspirations. Yet, in the absence of a national Multi-Listing Service (MLS) in Mexico, determining the true worth of my home has been a challenge. This project is my quest to uncover the hidden value of my home, to understand its worth in the context of the Ensenada real estate market.

Why now? Why this project? The answer lies in the desire to bridge the gap between the physical and the digital, to bring clarity to the real estate landscape that has, until now, been shrouded in mystery. By leveraging the power of web scraping, I aim to dive into the digital archives of point2homes.com, a decent sized listing service, and build a accurate enough model to track values.

This journey is not just about numbers and statistics; it's about uncovering the stories behind each property listing, the dreams and aspirations of those who call Ensenada home. And mostly understanding the market cycle about seeing how much cash I can make. 

Join me as I embark on this exciting journey, as I dive into the depths of the Mexican real estate market, uncovering the hidden gems that lie within. Together, we will bring the real estate landscape of Ensenada into the light, making it accessible and understandable to all. And in doing so, we will uncover the true worth of my home, a testament to the beauty and potential of Ensenada.


### First lets import the libraries we will be using


In [1]:
from bs4 import BeautifulSoup
import requests
import csv

We will use bs4 to call the website and return all of the html which will be stored in memory. I just looked up my user agent and copied it from the internet. 

also I will not bother to clean up and polish my notebooks. I really appreciate the thought process and the code tests when I am learning. I will not be using this for any other purpose than to teach others.  So if you are reading this, I hope you are learning something. 

Data Science is not a linear process its messy and you will make mistakes.



In [2]:
URL = f'https://www.point2homes.com/MX/Real-Estate-Listings/Baja-California/Ensenada.html'
headers = {
        'Accept': 'application/json, text/javascript',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
        }


In [3]:
# Get the page
page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, 'html.parser')


In [4]:
# lets look at the reposnse form the website.
print(soup.prettify())


<!DOCTYPE html>
<html lang="en-US">
 <head>
  <title>
   Just a moment...
  </title>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta content="IE=Edge" http-equiv="X-UA-Compatible"/>
  <meta content="noindex,nofollow" name="robots"/>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <style>
   *{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131}button,html{font-family:system-ui,-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica Neue,Arial,Noto Sans,sans-serif,Apple Color Emoji,Segoe UI Emoji,Segoe UI Symbol,Noto Color Emoji}@media (prefers-color-scheme:dark){body{background-color:#222;color:#d9d9d9}body a{color:#fff}body a:hover{color:#ee730a;text-decoration:underline}body .lds-ring div{border-color:#999 transparent transparent}body .font-red{color:#b20f03}body .big-button,body .pow-button{background-color:#4693ff;color:#1d1d1d}body #challenge-success-text{background-im

### Oh no we were blocked immediatly, javascript must be enabled to view the website.
Most companies have javascript scrolling and bot detection. We will use selenium to bypass this.
Selenium will open up a browser and scroll for us sp we can use it to scrape the website.
You may have to manually accept cookies and scroll down to load all of the listings. Once that is done we will take another look at the soup.
If you havent figured it out by now the soup is thehtml request from the website. we make one call and then store the html in memory.



In [5]:
from selenium import webdriver


In [6]:
driver = webdriver.Firefox()
driver.get(URL)
soup = BeautifulSoup(driver.page_source, 'html.parser')

In [7]:
print(soup.prettify()[:1000])

# Close the browser
driver.quit()

<html lang="en">
 <head>
  <script async="" src="https://matomo.mgmt.sharks.cloud/js/container_88C40TBZ.js">
  </script>
  <script async="" src="https://www.clarity.ms/s/0.7.26/clarity.js">
  </script>
  <script async="" src="https://www.clarity.ms/tag/kk321qb0ir?ref=gtm2">
  </script>
  <script async="" src="https://www.googletagmanager.com/gtag/js?id=G-XBM0745T8C&amp;l=dataLayer&amp;cx=c" type="text/javascript">
  </script>
  <script src="https://securepubads.g.doubleclick.net/pagead/managed/js/gpt/m202403270101/pubads_impl_page_level_ads.js">
  </script>
  <title>
   Ensenada, Baja California, Mexico Real Estate &amp; Homes for Sale | Point2
  </title>
  <link crossorigin="" href="https://cdn.point2homes.com" rel="preconnect"/>
  <link crossorigin="" href="https://mediavault.point2.com" rel="preconnect"/>
  <link href="https://www.point2homes.com/MX/Real-Estate-Listings/Baja-California/Ensenada.html" hreflang="en" rel="alternate"/>
  <link href="https://www.point2homes.com/ES/MX/Rea

### Success! we have the html of the website. Now we will use beautiful soup to parse the html and extract the data we need.
For this step i will usually pull up the inspect in chrome developer tools. use the selector tool to find the class or id of the element i want to scrape.
Then i will use the soup.select() method to get the data i need.



![Alt text](shot1.png)


In [8]:
# extract address-container
address_container = soup.find_all('div', class_='address-container')
# print 500 characters of the address_container
print(address_container[0].prettify()[:500])



<div class="address-container" data-address="Privada Toledo 1536, Ensenada, Baja California" title="Residential Property for sale in Privada Toledo 1536, Ensenada, Baja California">
 Privada Toledo 1536, Ensenada, Baja California
</div>



Title address seems to have some interesting data. the title and address are csv seperated. we could split this into multiple columns later. the last two will be city and state and then we can assume the remaining columns are the title of the listing. lets keep going and see what we can find.


In [9]:
# placeholder : extract title, address city, state from address_container

In [10]:
# inside div class characteristics-cnt extract beds, baths, sqft, price
characteristics_cnt = soup.find_all('div', class_='characteristics-cnt')
print(characteristics_cnt[0].prettify()[:500])


<div class="characteristics-cnt">
 <ul>
  <li class="ic-beds" data-label="Beds">
   <strong>
    3
   </strong>
   <span class="gray normal-lbl">
    Beds
   </span>
   <span class="gray short-lbl">
    Bds
   </span>
  </li>
  <li class="ic-baths" data-label="Baths">
   <strong>
    2
   </strong>
   <span class="gray normal-lbl">
    Baths
   </span>
   <span class="gray short-lbl">
    Ba
   </span>
  </li>
  <li class="ic-sqft" data-label="Sqft">
   <strong>
    1,829.86
   </strong>
   <spa


#### Web scraping pro tip. 

This is kind of  a tedious process and I'm not very patient so i took a sample of the html and gave it to claude.ai to identify the div and classes etc that i need to extract the information. Claude is amazing for this sort of task and you just have to be a little sneaky about how you use it. I will use the information to extract the data i need.

In [23]:
# Find all the listing elements
listings = soup.select('div.item-cnt')

# Iterate over each listing
for listing in listings:
    # Extract the address
    address_element = listing.select_one('div.address-container')
    address = address_element['data-address'] if address_element else None

    # Extract the number of bedrooms
    beds_element = listing.select_one('li.ic-beds strong')
    beds = beds_element.text if beds_element else None

    # Extract the number of bathrooms
    baths_element = listing.select_one('li.ic-baths strong')
    baths = baths_element.text if baths_element else None

    # Extract the square footage
    sqft_element = listing.select_one('li.ic-sqft strong')
    sqft = sqft_element.text if sqft_element else None

    # Extract the lot size
    lot_size_element = listing.select_one('li.ic-lotsize strong')
    lot_size = lot_size_element.text if lot_size_element else None

    # Extract the property type
    property_type_element = listing.select_one('li.ic-proptype')
    property_type = property_type_element.text if property_type_element else None

    # Extract the price
    price_element = listing.select_one('div.price')
    price = price_element.text if price_element else None

    # Extract the listing agent name and company
    agent_name_element = listing.select_one('div.agent-name a')
    agent_name = agent_name_element.text if agent_name_element else None

    agent_company_element = listing.select_one('div.agent-company')
    agent_company = agent_company_element.text if agent_company_element else None

    # # Print the extracted information
    # print(f'Address: {address}')
    # print(f'Beds: {beds}')
    # print(f'Baths: {baths}')
    # print(f'Square Feet: {sqft}')
    # print(f'Lot Size: {lot_size}')
    # print(f'Property Type: {property_type}')
    # print(f'Price: {price}')
    # print(f'Listing Agent: {agent_name}, {agent_company}')
    # print('---')
    

In [12]:
# count the number of listings in soup
print(f'Total number of listings: {len(listings)}')


Total number of listings: 24


### Excellent! 
This si a decent start. but we are missing a lot of data. We will need to do do 2 things. First scroll down and load all of the listings with selenium download that code and then we need to find the next page button click  it scroll and store the data for all of the pages.





In [18]:
# to be continued

[]
